In [0]:
%pip install prophet

In [0]:
from prophet import Prophet

In [0]:
date_dim_df = spark.read.table('mini_project.gold_layer.date_dim')

display(date_dim_df)

In [0]:
sales_order_df = spark.read.table('mini_project.gold_layer.sales_order')

display(sales_order_df)

In [0]:
from pyspark.sql import functions as F

# 1) Prepare sales line amounts + normalize order date to DATE
sales_prepped_df = (
    sales_order_df
    .filter(F.col("status") == "Shipped")
    .withColumn("order_date_ts", F.to_timestamp("order_date"))
    .withColumn("order_date", F.to_date("order_date_ts"))
    .withColumn(
        "line_sales_amount",
        F.col("order_qty").cast("double")
        * F.col("unit_price").cast("double")
        * (F.lit(1.0) - F.col("unit_price_discount").cast("double"))
    )
)

# 2) Aggregate sales to daily level first
daily_sales_df = (
    sales_prepped_df
    .groupBy("order_date")
    .agg(F.sum("line_sales_amount").alias("daily_sales"))
)

# 3) Build month calendar from date_dim (one row per month)
#    ds = first day of month (Prophet-friendly)
month_calendar_df = (
    date_dim_df
    .withColumn("date", F.to_date("date"))
    .withColumn("ds", F.to_date(F.date_trunc("month", F.col("date"))))
    .select("ds", "year", "quarter", "month", "month_name")
    .dropDuplicates(["ds"])
    .orderBy("ds")
)

# 4) Join daily sales to date_dim, then roll up to month
#    This ensures months with no sales still exist
monthly_prophet_spark_df = (
    date_dim_df
    .withColumn("date", F.to_date("date"))
    .join(
        daily_sales_df,
        date_dim_df["date"] == daily_sales_df["order_date"],
        how="left"
    )
    .filter(F.col("year")!=2025)
    .withColumn("daily_sales", F.coalesce(F.col("daily_sales"), F.lit(0.0)))
    .withColumn("ds", F.to_date(F.date_trunc("month", F.col("date"))))
    .groupBy("ds")
    .agg(F.sum("daily_sales").alias("y"))
    .orderBy("ds")
)

display(monthly_prophet_spark_df)

In [0]:
from prophet import Prophet

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

model.fit(monthly_prophet_spark_df.toPandas())


In [0]:
future_pd = model.make_future_dataframe(
    periods=12,
    freq="MS",
    include_history=True
)

# predict over the dataset
forecast_pd = model.predict(future_pd)

In [0]:
fig = model.plot(forecast_pd)
fig2 = model.plot_components(forecast_pd)